Parte 1. Carga y Exploracion de datos 

In [1]:

import pandas as pd 
import matplotlib.pyplot as plt
import math
import random
import numpy as np

Cargamos el CVS con pd.read_csv

In [2]:
df = pd.read_csv("proyecto_final_pacientes_biomedicos.csv")

#Mostrar las primeras filas del DataFrame
##.head muestra las primeras filas del DataFrame, lo que nos permite tener una idea de cómo están organizados los datos y qué tipo de información contienen
print(df.head())

  id_paciente  edad sexo    grupo_clinico  fc_lpm  ruido_ecg estado_eeg  \
0        P001    21    F  control_estable      64      0.018   relajado   
1        P002    24    M  control_estable      68      0.020   relajado   
2        P003    27    F  control_estable      72      0.018     alerta   
3        P004    29    M  control_estable      75      0.022     alerta   
4        P005    31    F  control_estable      61      0.015   relajado   

   ruido_eeg  factor_contraccion  pa_sistolica  pa_diastolica  spo2_media  \
0      0.040                0.22           112             72        98.7   
1      0.045                0.28           118             76        98.3   
2      0.040                0.30           120             78        97.9   
3      0.050                0.35           122             80        98.1   
4      0.035                0.18           110             70        99.0   

   duracion_segundos prioridad  \
0                 10      baja   
1                 

In [3]:
#Mostrar dimensiones de la tabla 
## .shape nos permite conocer las dimesiones del data, numeros de fila y columnas 

df.shape


(24, 15)

In [4]:
#Tipos de datos de cada columna
## .dtypes nos ayuda a identificar el tipo de datos que tiene cada columna

df.dtypes

id_paciente               str
edad                    int64
sexo                      str
grupo_clinico             str
fc_lpm                  int64
ruido_ecg             float64
estado_eeg                str
ruido_eeg             float64
factor_contraccion    float64
pa_sistolica            int64
pa_diastolica           int64
spo2_media            float64
duracion_segundos       int64
prioridad                 str
nota_clinica              str
dtype: object

In [5]:
#Conteo de pacientes por Grupo clinico 
## .value_counts nos perimte contar la frecuencia de cada valor unico en la columna
df['grupo_clinico'].value_counts()


grupo_clinico
control_estable       6
taquicardia_estres    6
hipertension          6
hipoxemia_fatiga      6
Name: count, dtype: int64

In [7]:
#Verificar si hay valores faltantes 
## .isna nos identifica los valores faltantes mientras que el .sum nos dice el numero de valores que faltan por cada columna
df.isna().sum()

id_paciente           0
edad                  0
sexo                  0
grupo_clinico         0
fc_lpm                0
ruido_ecg             0
estado_eeg            0
ruido_eeg             0
factor_contraccion    0
pa_sistolica          0
pa_diastolica         0
spo2_media            0
duracion_segundos     0
prioridad             0
nota_clinica          0
dtype: int64

Parte 2. Preparacion de la base con pandas

In [6]:
#Columna de pulso de presion 
df['pulso_presion'] = df['pa_sistolica'] - df['pa_diastolica']

#Columna de Hipoxemia (spo2_media < 92)
df['hipoxemia'] = df['spo2_media'] < 92

#Columna de hipertension (sistolica >= 140 o diastolica >= 90)
df['hipertencion'] = (df['pa_sistolica'] >= 140) | (df['pa_diastolica'] >= 90)

#Columna de taquicardia (fc_lpm > 100)
df['taquicardia'] = df['fc_lpm'] > 100


In [7]:
#visualizar columnas agregadas 
df.head()

,id_paciente,edad,sexo,grupo_clinico,fc_lpm,ruido_ecg,estado_eeg,ruido_eeg,factor_contraccion,pa_sistolica,pa_diastolica,spo2_media,duracion_segundos,prioridad,nota_clinica,pulso_presion,hipoxemia,hipertencion,taquicardia
0,P001,21,F,control_estable,64,0.018,relajado,0.040,0.22,112,72,98.7,10,baja,Paciente de referencia sin alteraciones clinicas.,40,False,False,False
1,P002,24,M,control_estable,68,0.020,relajado,0.045,0.28,118,76,98.3,10,baja,Perfil fisiologico normal con senales estables.,42,False,False,False
2,P003,27,F,control_estable,72,0.018,alerta,0.040,0.30,120,78,97.9,10,baja,Paciente sano en actividad ligera.,42,False,False,False
3,P004,29,M,control_estable,75,0.022,alerta,0.050,0.35,122,80,98.1,12,baja,Frecuencia cardiaca normal alta con buena oxig...,42,False,False,False
4,P005,31,F,control_estable,61,0.015,relajado,0.035,0.18,110,70,99.0,10,baja,Paciente en reposo con presion arterial normal.,40,False,False,False


Preguntas preparacion de la base con pandas 

In [8]:
#.sum contara los true por los datos boleanos 
#1. ¿Cuantos pacientes tienen hipoxemia?.
total_hipoxemia = df['hipoxemia'] .sum()
print(f"1.Total de pacientes con hipoxemia es:{total_hipoxemia}")


#2. ¿Cuantos pacientes tienen taquicardia?.
total_taquicardia = df['taquicardia'] .sum()
print(f"2.Total de pacientes con taquicardia es:{total_taquicardia}")

#3. ¿Cual grupo presenta mayor presion sistolica promedio?.
presion_sistolica_promedio = df.groupby('grupo_clinico')['pa_sistolica'].mean()
print("3. Presion sistolica promedio por grupo clinico:")
print(presion_sistolica_promedio)


1.Total de pacientes con hipoxemia es:5
2.Total de pacientes con taquicardia es:10
3. Presion sistolica promedio por grupo clinico:
grupo_clinico
control_estable       117.666667
hipertension          161.333333
hipoxemia_fatiga      125.000000
taquicardia_estres    136.166667
Name: pa_sistolica, dtype: float64


 Parte 3. Reutilizacion de funciones del notebook 00

In [9]:
#Funcion para modelar ondas cardiacas 
def gaussiana(t, amplitud , centro, ancho):
    return amplitud * np.exp(-((t - centro) ** 2) / (2 * ancho ** 2))

In [10]:
#Funcion nivel de contraccion 
def nivel_contraccion(t, contraccion=0.5, fs=2000):
    amplitud = contraccion * 2.0  
    señal = random.gauss(0, amplitud)
    
    # Componente de baja frecuencia (artefacto de movimiento)
    
    señal += 0.1 * math.sin(2 * math.pi * 3 * t)
    return señal


In [11]:
#Adaptar el EMG sintetico para incluir el nivel de contraccion
def emg_sintetico(t, contraccion=1.0):
    # Simular un EMG con ruido y variabilidad
    frecuencia_base = 10  # Frecuencia base del EMG
    ruido = np.random.normal(0, 0.1, size=t.shape)  # Ruido aleatorio
    emg = contraccion * np.sin(2 * np.pi * frecuencia_base * t) + ruido
    return emg

In [ ]:
#Funcion para EEG sintetico
def eeg_sintetico(t, estado="relajado", ruido=0.05):
    pi2 = 2 * math.pi
    
    # Amplitudes por estado (cada estado activa distintas bandas)
    amplitudes = {
        "sueño":       {"delta": 1.5, "theta": 0.5, "alfa": 0.2, "beta": 0.05},
        "relajado":    {"delta": 0.3, "theta": 0.4, "alfa": 1.0, "beta": 0.3},
        "alerta":      {"delta": 0.1, "theta": 0.2, "alfa": 0.3, "beta": 0.8},
        "concentrado": {"delta": 0.1, "theta": 0.1, "alfa": 0.1, "beta": 1.2},
    }
    amp = amplitudes.get(estado, amplitudes["relajado"])
    
    señal = (
        amp["delta"] * math.sin(pi2 * 2   * t) +    # δ: 2 Hz
        amp["theta"] * math.sin(pi2 * 6   * t) +    # θ: 6 Hz
        amp["alfa"]  * math.sin(pi2 * 10  * t) +    # α: 10 Hz
        amp["beta"]  * math.sin(pi2 * 20  * t)      # β: 20 Hz
    )
    señal += random.gauss(0, ruido)
    return señal * 50   # escalar a μV

In [13]:
#Funcion para ECG sintetico
def ecg_sintetico(t, fc=72, ruido=0.02):
    periodo = 60.0 / fc       # duración de un ciclo cardíaco [s]
    t_ciclo = (t % periodo) / periodo   # posición normalizada [0, 1] en el ciclo

    señal = 0.0
    # Onda P (despolarización auricular)
    señal += gaussiana(t_ciclo, 0.15, 0.18, 0.025)
    # Onda Q (inicio de despolarización ventricular — negativa)
    señal -= gaussiana(t_ciclo, 0.10, 0.42, 0.010)
    # Onda R (despolarización ventricular — pico principal)
    señal += gaussiana(t_ciclo, 1.00, 0.46, 0.008)
    # Onda S (despolarización septal — negativa)
    señal -= gaussiana(t_ciclo, 0.22, 0.50, 0.008)
    # Onda T (repolarización ventricular)
    señal += gaussiana(t_ciclo, 0.30, 0.68, 0.035)

    # Ruido fisiológico 
    señal += random.gauss(0, ruido)
    return señal

In [14]:
#Funcion para modelar presion arterial
def presion_arterial(t, sistolica = 120, diastolica =80, fc= 72):
    periodo = 60.0 / fc
    t_norm = (t % periodo) / periodo
    
    # Pico sistólico
    pico = (sistolica - diastolica) * gaussiana(t_norm, 1.0, 0.25, 0.08)
    # Muesca dicrota (rebote de la válvula aórtica)
    muesca = (sistolica - diastolica) * 0.15 * gaussiana(t_norm, 1.0, 0.55, 0.04)
    
    return diastolica + pico + muesca + random.gauss(0, 0.5)


In [15]:
#Funcion para modelar Sp02 
def spo2_ppg(t, spo2_media=98.5, fc=72):
    periodo = 60.0 / fc
    
    # Variación pulsátil del SpO₂ (muy pequeña en condiciones normales)
    
    variacion = 0.5 * math.sin(2 * math.pi * t / periodo)
   
    # Modulación respiratoria lenta
    
    modulacion_resp = 0.3 * math.sin(2 * math.pi * 0.25 * t)   # 15 resp/min
    return spo2_media + variacion + modulacion_resp + random.gauss(0, 0.1)


In [16]:
def t_vector(tiempo_total=10, fs=2000):
    return np.linspace(0, tiempo_total, int(tiempo_total * fs), endpoint=False) 

Parte 4. Generacion de senales por paciente

In [23]:
def generar_senales_paciente(fila):
    # 1. Crear el vector de tiempo 
    t = t_vector(tiempo_total=10, fs=500) 
    
    # 2. Generar cada señal usando tus funciones definidas
    
    sig_ecg  = ecg_sintetico(t, fc=fila['fc_lmp'])
    
    sig_eeg  = eeg_sintetico(t, estado="relajado") # o el estado que venga en la fila
    
    sig_emg  = emg_sintetico(t, contraccion=fila['nivel_emg'])
    
    sig_pa   = presion_arterial(t, sistolica=fila['pa_sistolica'], 
                                   diastolica=fila['pa_diastolica'], 
                                   fc=fila['fc_lpm'])
    
    sig_spo2 = spo2_ppg(t, spo2_media=fila['spo2'], 
                           fc=fila['fc_lpm  '])
    
    # 3. Empaquetar todo en el DataFrame con los nombres solicitados
    df_resultado = pd.DataFrame({
        'tiempo_s': t,
        'ecg_mV': sig_ecg,
        'eeg_uV': sig_eeg,
        'emg_mV': sig_emg,
        'pa_mmHg': sig_pa,
        'spo2_pct': sig_spo2
    })
    
    return df_resultado

In [24]:

fila_paciente = df.iloc[0]

# Generamos el DataFrame de señales
df_senales_final = generar_senales_paciente(fila_paciente)

# Mostrar resultado
df_senales_final.head()

KeyError: 'fc_lmp'

In [19]:

paciente_data = df.iloc[0]  # Tomamos la fila del primer paciente
df_senales_p1 = generar_senales_paciente(paciente_data)

# Visualizamos el resultado
df_senales_p1.head()

KeyError: 'fc_lmp'

## 5-Pacientes a analizar